# bench_imai_var — ArtiFact Vision Benchmark

Train and evaluate vision models on ArtiFact dataset (2.5M images, 25 generators) with cross-dataset generalization testing on CIFAKE and Shoes datasets.

## Requirements

**GCP VM with GPU** — Run `./gcp_setup.sh` first (see `GCP_REMOTE_SETUP.md`)

## Configuration

| Setting | Value |
|---------|-------|
| **VM** | n1-standard-16 (16 vCPUs, 60GB RAM) + T4 GPU |
| **Batch Size** | 64 (optimized for 16 vCPUs) |
| **Training Data** | ArtiFact (200×200, 25 generators, 15% sample) |
| **Test Datasets** | ArtiFact, CIFAKE, Shoes |
| **Total Cost** | ~$2.80 (2.5 hours @ $1.11/hr) |

## Quick Start

**Before running this notebook:**
1. Provision VM: `./gcp_setup.sh`
2. Connect via VS Code Remote SSH
3. Set credentials in cell below:
   ```python
   os.environ["KAGGLE_USERNAME"] = "your_username"
   os.environ["KAGGLE_KEY"] = "your_api_key"
   ```

In [ ]:
import os
import sys
from pathlib import Path

# Verify GCP VM environment
import os
os.environ["GCP_REMOTE_GPU"] = "1"
os.environ["KAGGLE_USERNAME"] = os.environ.get("KAGGLE_USERNAME", "orel_mazor")
os.environ["KAGGLE_KEY"] = os.environ.get("KAGGLE_KEY")

NOTEBOOK_ROOT = Path.cwd()
print("✅ GCP VM environment detected")
print("✅ Kaggle credentials configured")
print(f"📁 Working directory: {NOTEBOOK_ROOT}")

## 1. Verify Environment

Check Python environment and install dependencies if needed.

In [ ]:
# Find repository root (should be already cloned by gcp_setup.sh)
REPO_DIR = None

# Check current directory and parent
if (NOTEBOOK_ROOT / "requirements.txt").exists():
    REPO_DIR = NOTEBOOK_ROOT
elif (NOTEBOOK_ROOT.parent / "requirements.txt").exists():
    REPO_DIR = NOTEBOOK_ROOT.parent
else:
    # Search common locations
    possible_paths = [
        Path.home() / "bench_research_ml_project",
        Path("/workspace/bench_research_ml_project"),
    ]
    for path in possible_paths:
        if path.exists() and (path / "requirements.txt").exists():
            REPO_DIR = path
            break

if REPO_DIR is None:
    raise FileNotFoundError(
        "❌ Repository not found!\n"
        "The gcp_setup.sh script should have cloned it.\n"
        "Please run: git clone https://github.com/oremaz/bench_research_ml_project"
    )

print(f"✅ Repository found: {REPO_DIR}")

# Verify and install dependencies
import subprocess

def pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    subprocess.check_call(cmd)

print("📦 Verifying dependencies...")
try:
    import timm
    import kaggle
    import rich
    print("✅ All dependencies installed")
except ImportError:
    print("📦 Installing missing dependencies...")
    pip_install("-r", str(REPO_DIR / "requirements.txt"))
    pip_install("numpy==1.24.3", "scipy==1.10.1", "scikit-learn==1.3.0")
    pip_install("timm==1.0.9", "kaggle", "rich")
    print("✅ Dependencies installed")

In [ ]:
# Change to ml_pipeline directory
os.chdir(REPO_DIR / "ml_pipeline")
print(f"✅ Working directory: {Path.cwd()}")

# Download FatFormer CLIP weights if needed
from pathlib import Path
import urllib.request

FATFORMER_DIR = Path("third_party/FatFormer")
PRETRAINED_DIR = FATFORMER_DIR / "pretrained"
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_URL = "https://openaipublic.azureedge.net/clip/models/b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt"
MODEL_PATH = PRETRAINED_DIR / "ViT-L-14.pt"

if not MODEL_PATH.exists():
    print(f"⬇️  Downloading FatFormer CLIP checkpoint...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(f"✅ Downloaded to {MODEL_PATH}")
else:
    print(f"✅ CLIP checkpoint exists: {MODEL_PATH}")

## 2. Imports and deterministic utilities
Everything important (models, metrics, benchmarking) is imported from the shared
library so we avoid redefining models or augmentations inside the notebook.


In [ ]:
import os
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.vision_models import MODEL_REGISTRY as VISION_MODELS
from pipelines_torch.base import SimplePredictor
from utils.metrics import METRIC_REGISTRY
from utils.utils import load_model

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Verify GPU availability (critical for training)
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU detected!\n"
        "This notebook requires a GPU for training.\n"
        "Check your GCP VM configuration:\n"
        "1. VM should have GPU attached (T4, V100, or A100)\n"
        "2. NVIDIA drivers should be installed\n"
        "3. Run: nvidia-smi (should show GPU info)\n\n"
        "See GCP_REMOTE_SETUP.md for troubleshooting."
    )

print(f"✅ CUDA version: {torch.version.cuda}")
print(f"✅ GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"    Memory: {props.total_memory / 1e9:.2f} GB")
    print(f"    Compute: {props.major}.{props.minor}")

## 3. Download and prepare ArtiFact Dataset
The ArtiFact dataset contains 2.5M images (965K real + 1.5M fake) from 25 different 
generators at 200×200 resolution. This provides much more generator diversity than CIFAKE.


In [ ]:
from pathlib import Path
from utils.kaggle_utils import ensure_kaggle_dataset

# Kaggle dataset: https://www.kaggle.com/datasets/awsaf49/artifact-dataset
KAGGLE_ARTIFACT = "awsaf49/artifact-dataset"

# ✅ Use the actual extracted top-level folder name from the dataset zip:
EXPECTED_TOPDIR = "artifact-dataset"

ARTIFACT_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_ARTIFACT,
    local_dir=Path("data/artifact-real-fake"),
    description="ArtiFact Real and Fake Images Dataset",
    kaggle_subdir=EXPECTED_TOPDIR,
)

if ARTIFACT_DIR.exists():
    print(f"✅ ArtiFact dataset available at {ARTIFACT_DIR}")
else:
    print(f"⚠️ ArtiFact dataset missing at {ARTIFACT_DIR}")

In [ ]:
# ArtiFact dataset uses 200x200 images
IMG_SIZE = 200
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ArtiFact dataset structure - look for train/test or use the directory structure
def find_imagefolder_split(base_dir: Path) -> Path:
    """Find the appropriate ImageFolder structure in ArtiFact dataset"""
    base_dir = Path(base_dir)
    
    # Check if base is already an ImageFolder root
    def is_imagefolder_dir(path: Path) -> bool:
        path = Path(path)
        if not path.is_dir():
            return False
        subdirs = [p for p in path.iterdir() if p.is_dir()]
        if len(subdirs) < 2:
            return False
        return any(any(child.is_file() for child in d.iterdir()) for d in subdirs)
    
    if is_imagefolder_dir(base_dir):
        return base_dir
    
    # Look for common split names
    preferred = ("train", "test", "validation", "val", "eval")
    for name in preferred:
        for variant in {name, name.upper(), name.capitalize()}:
            candidate = base_dir / variant
            if is_imagefolder_dir(candidate):
                return candidate
    
    # Look for real/fake structure
    for candidate in base_dir.glob("*"):
        if candidate.is_dir():
            real_dir = candidate / "real" 
            fake_dir = candidate / "fake"
            if real_dir.exists() and fake_dir.exists() and is_imagefolder_dir(candidate):
                return candidate
    
    # General recursive search
    for candidate in sorted(base_dir.rglob("*")):
        if is_imagefolder_dir(candidate):
            return candidate
    
    raise ValueError(f"Could not locate an ImageFolder split inside {base_dir}")

# Find the training data structure
train_split = find_imagefolder_split(ARTIFACT_DIR)
print(f"Using ArtiFact data from: {train_split}")

# Create datasets
train_ds = datasets.ImageFolder(train_split, transform=transform)

class_names = train_ds.classes
num_classes = len(class_names)
print(f"Classes: {class_names}")
print(f"Total training samples: {len(train_ds)}")

In [ ]:
def dataset_to_numpy(dataset: datasets.ImageFolder):
    tensors = [img for img, _ in dataset]
    X = torch.stack(tensors).numpy()
    y = np.array(dataset.targets, dtype=np.int64)
    return X.astype(np.float32), y

X_all, y_all = dataset_to_numpy(train_ds)
SAMPLE_FRACTION = 0.15         # 15 % sampling
print(f"Original ArtiFact size: {len(X_all)} images")

# Stratified sampling keeps the real/fake balance intact
X_all, _, y_all, _ = train_test_split(
    X_all,
    y_all,
    train_size=SAMPLE_FRACTION,
    random_state=SEED,
    stratify=y_all,
)
print(f"Training data shape: {X_all.shape}")
print(f"Class distribution: {np.bincount(y_all)}")

## 4. Configure `BenchmarkRunner` with Optimized Epochs
We use different epoch counts based on model complexity:
- Complex models (transformers, large CNNs): 5 epochs
- Simple models (adaptive_cnn, residual_cnn): 30 epochs  
- Remove simple_cnn as requested


In [ ]:
# Select models (excluding simple_cnn and problematic ones)
models = []
for model in VISION_MODELS: 
    if model not in ["qwen2_vl_qlora", "fatformer_official", "simple_cnn"]: 
        models.append(model)

model_configs = []
epochs = {}

# Define epoch counts based on model complexity
for name in models:
    if name not in VISION_MODELS:
        raise KeyError(f"{name} is not registered in pipelines_torch.vision_models")
    
    model_configs.append({
        "name": name,
        "class": VISION_MODELS[name],
        "params": {"num_classes": num_classes},
    })
    
    # Assign epochs based on model complexity
    if name in ["adaptive_cnn", "residual_cnn"]:
        epochs[name] = 30  # Simple models get more epochs
    else:
        epochs[name] = 5   # Complex models get fewer epochs

print(f"Training {len(models)} models with epochs: {epochs}")

# Optimized configuration for GCP VM
BATCH_SIZE = 64  # Optimized for n1-standard-16 (16 vCPUs)
NUM_WORKERS = 12  # Parallel data loading (adjust based on vCPUs)

runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],
    task_type="classification",
    device=DEVICE,
    epochs=epochs,
    batch_size=BATCH_SIZE,
    early_stopping=None,
    use_class_weights=True,
    use_kfold=False,
    learning_rate=3e-4,
    path_start="bench_imai_var",
    random_state=SEED,
)

print(f"\n⚙️  Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Num workers: {NUM_WORKERS}")
print(f"  Device: {DEVICE}")

results_df = runner.run(X_all, y_all)
results_df

In [ ]:
from pathlib import Path
import tarfile

results_root = Path("results")
experiment_dir = results_root / "bench_imai_var"
archive_path = results_root / "bench_imai_var_models.tar.gz"

if not experiment_dir.exists():
    print(f"⚠️  Expected results directory {experiment_dir} not found.")
    print("Run the training cell first.")
else:
    results_root.mkdir(exist_ok=True)
    with tarfile.open(archive_path, "w:gz") as tar:
        for file_path in experiment_dir.rglob("*.pt"):
            tar.add(file_path, arcname=file_path.relative_to(results_root))
    
    print(f"✅ Created archive at {archive_path.resolve()}")
    print(f"   Size: {archive_path.stat().st_size / 1e6:.1f} MB")
    print(f"\nDownload with:")
    print(f"  scp your-vm:{archive_path.resolve()} .")

## 5. Evaluation Infrastructure
Reuse the same evaluation infrastructure to test across all three datasets.


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset, TensorDataset
from typing import Callable, List, Optional, Dict, Any, Tuple, Union
import numpy as np
import random
from functools import partial
from collections.abc import Sized
from typing import cast
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize
from utils.utils import select_best_epoch

class Evaluate:
    """Base utilities shared across torch and sklearn pipelines."""

    def __init__(self) -> None:
        # These attributes should be set by subclasses
        self.model = None
        self.task_type = "classification"
        self.metrics = []
        self.batch_size = 32
        self.random_state = 42
        self.device = "cpu"

    def _default_selection_metric(self) -> str:
        """Return the metric name used to pick the best epoch."""
        return "roc_auc" if self.task_type == "classification" else "r2_score"

    def prepare_data_internal(
        self,
        X: np.ndarray,
        y: Optional[np.ndarray] = None,
        *,
        train: bool = True,
    ) -> DataLoader:
        """Subclasses must implement how raw arrays become loaders."""
        raise NotImplementedError("Subclasses must implement prepare_data_internal")

    # ----- Evaluation helpers -------------------------------------------------
    def evaluate(self, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        if X is None or y is None:
            return {}
        if hasattr(self, "model") and hasattr(self.model, "eval"):
            return self._evaluate_torch(X, y)
        return self._evaluate_sklearn(X, y)

    def _evaluate_torch(self, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        loader: DataLoader = self.prepare_data_internal(X, y, train=False)
        self.model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in loader:
                xb, yb = batch
                xb = xb.to(self.device)
                yb = yb.to(self.device)
                logits = self.model(xb)
                if self.task_type == "regression":
                    outputs = logits.squeeze(-1)
                else:
                    outputs = torch.softmax(logits, dim=1)
                preds.append(outputs.cpu())
                targets.append(yb.cpu())
        preds_np = torch.cat(preds).numpy()
        targets_np = torch.cat(targets).numpy()
        results: Dict[str, float] = {}
        for metric in self.metrics:
            key = getattr(metric, "name", None) or getattr(metric, "__name__", None) or str(metric)
            try:
                results[key] = float(metric(targets_np, preds_np))
            except Exception:
                results[key] = 0.0
        return results

    def _evaluate_sklearn(self, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        if not hasattr(self.model, "predict"):
            raise AttributeError(f"Model of type {type(self.model)} does not have a predict method.")
        y_pred = self.model.predict(X)
        results: Dict[str, float] = {}
        for metric in self.metrics:
            key = getattr(metric, "name", None) or getattr(metric, "__name__", None) or str(metric)
            try:
                results[key] = float(metric(y, y_pred))
            except Exception:
                results[key] = 0.0
        return results

    # ----- Prediction helpers -------------------------------------------------
    def predict(self, X: np.ndarray) -> np.ndarray:
        if hasattr(self, "model") and hasattr(self.model, "eval"):
            return self._predict_torch(X)
        return self._predict_sklearn(X)

    def _predict_torch(self, X: np.ndarray) -> np.ndarray:
        loader: DataLoader = self.prepare_data_internal(X, train=False)
        self.model.eval()
        preds = []
        with torch.no_grad():
            for batch in loader:
                xb = batch[0] if isinstance(batch, (list, tuple)) else batch
                xb = xb.to(self.device)
                logits = self.model(xb)
                if self.task_type == "regression":
                    preds.append(logits.squeeze(-1).cpu())
                else:
                    if logits.ndim == 1:
                        logits = logits.unsqueeze(-1)
                    if logits.shape[1] == 1:
                        probs = torch.sigmoid(logits).squeeze(-1)
                        preds.append((probs > 0.5).long().cpu())
                    else:
                        preds.append(torch.softmax(logits, dim=1).argmax(dim=1).cpu())
        return torch.cat(preds).numpy()

    def _predict_sklearn(self, X: np.ndarray) -> np.ndarray:
        if hasattr(self.model, "predict"):
            return self.model.predict(X)
        raise AttributeError(f"Model of type {type(self.model)} does not have a predict method.")

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        if self.task_type != "classification":
            raise ValueError("predict_proba is only available for classification tasks")
        if hasattr(self, "model") and hasattr(self.model, "eval"):
            return self._predict_proba_torch(X)
        return self._predict_proba_sklearn(X)

    def _predict_proba_torch(self, X: np.ndarray) -> np.ndarray:
        loader: DataLoader = self.prepare_data_internal(X, train=False)
        self.model.eval()
        probs_list = []
        with torch.no_grad():
            for batch in loader:
                xb = batch[0] if isinstance(batch, (list, tuple)) else batch
                xb = xb.to(self.device)
                logits = self.model(xb)
                if logits.ndim == 1:
                    logits = logits.unsqueeze(-1)
                if logits.shape[1] == 1:
                    pos = torch.sigmoid(logits).squeeze(-1)
                    probs = torch.stack((1 - pos, pos), dim=1)
                else:
                    probs = torch.softmax(logits, dim=1)
                probs_list.append(probs.cpu())
        return torch.cat(probs_list).numpy()

    def _predict_proba_sklearn(self, X: np.ndarray) -> np.ndarray:
        if hasattr(self.model, "predict_proba"):
            return self.model.predict_proba(X)
        raise ValueError("Model does not support predict_proba")

class SimplePredictor(Evaluate):
    """
    Lightweight class for making predictions with trained models.
    Minimal setup required - just provide the model and basic info.
    """
    
    def __init__(self, model, task_type: str = "classification", device: str = "cpu", batch_size: int = 32):
        super().__init__()
        self.model = model
        self.task_type = task_type
        self.device = device
        self.batch_size = batch_size
        self.random_state = 42
        self.metrics = []
        
        # Move PyTorch models to device
        if hasattr(model, 'to'):
            self.model = model.to(device)
        
    def prepare_data_internal(self, X: np.ndarray, y: Optional[np.ndarray] = None, train: bool = True) -> DataLoader:
        """Prepare data for PyTorch models, return None for sklearn models."""
        if hasattr(self, 'device') and hasattr(self.model, 'eval'):
            # PyTorch model - create DataLoader
            X_tensor = torch.tensor(X, dtype=torch.float32)
            if y is not None:
                y_tensor = torch.tensor(y, dtype=torch.float32 if self.task_type == "regression" else torch.long)
                dataset = TensorDataset(X_tensor, y_tensor)
            else:
                dataset = TensorDataset(X_tensor)
            
            return DataLoader(
                dataset, 
                batch_size=self.batch_size, 
                shuffle=False, # No need to shuffle for prediction
                num_workers=0,
                drop_last=False
            )
        else:
            # sklearn model - return None (data passed directly)
            return None

In [ ]:
def evaluate_saved_models(model_names: Iterable[str], X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    metric_map = {
        "accuracy": "accuracy",
        "f1_macro": "f1",
        "precision_macro": "precision",
        "recall_macro": "recall",
        "roc_auc": "roc_auc",
        "pr_auc": "pr_auc",
    }
    records = []
    for name in model_names:
        try:
            model = load_model(VISION_MODELS[name], name, {"num_classes": num_classes}, path_start="bench_imai_var")
        except FileNotFoundError:
            print(f"⚠️  Skipping {name}: checkpoint not found")
            continue
        print(f"Evaluating: {name}")
        predictor = SimplePredictor(model, task_type="classification", device=DEVICE, batch_size=BATCH_SIZE)
        probs = predictor.predict_proba(X)
        scores = {
            label: float(METRIC_REGISTRY[key](y, probs))
            for label, key in metric_map.items()
        }
        records.append({
            "model": name,
            **scores,
        })
    return pd.DataFrame.from_records(records)

## 6. Test on ArtiFact Test Split
First, let's create or find a test split from the ArtiFact dataset.


In [ ]:
# Try to find a test split in the ArtiFact dataset
# If not available, we'll create one from the training data
try:
    # Look for existing test split
    test_candidates = [
        ARTIFACT_DIR / "test",
        ARTIFACT_DIR / "Test", 
        ARTIFACT_DIR / "val",
        ARTIFACT_DIR / "validation"
    ]
    
    artifact_test_dir = None
    for candidate in test_candidates:
        if candidate.exists() and any(candidate.iterdir()):
            try:
                test_ds = datasets.ImageFolder(candidate, transform=transform)
                if len(test_ds) > 0:
                    artifact_test_dir = candidate
                    break
            except:
                continue
    
    if artifact_test_dir is not None:
        print(f"Found ArtiFact test split at: {artifact_test_dir}")
        test_ds = datasets.ImageFolder(artifact_test_dir, transform=transform)
        X_test_artifact, y_test_artifact = dataset_to_numpy(test_ds)
    else:
        # Create test split from training data (20% split)
        print("No test split found, creating 20% test split from training data")
        from sklearn.model_selection import train_test_split
        
        # Use stratified split to maintain class balance
        indices = np.arange(len(X_all))
        train_idx, test_idx = train_test_split(
            indices, test_size=0.2, random_state=SEED, stratify=y_all
        )
        
        X_test_artifact = X_all[test_idx]
        y_test_artifact = y_all[test_idx]
        
        # Update training data to exclude test samples
        X_all = X_all[train_idx]
        y_all = y_all[train_idx]
        
        print(f"Created test split: {len(X_test_artifact)} samples")
        print(f"Updated training data: {len(X_all)} samples")
        
except Exception as e:
    print(f"Error setting up test split: {e}")
    # Fallback: use 10% of training data as test
    test_size = len(X_all) // 10
    X_test_artifact = X_all[:test_size]
    y_test_artifact = y_all[:test_size]
    print(f"Fallback: Using {test_size} samples for testing")

print(f"ArtiFact test shape: {X_test_artifact.shape}")
print(f"ArtiFact test class distribution: {np.bincount(y_test_artifact)}")

In [ ]:
# Test all models on ArtiFact test split
models_to_test = []
for model in VISION_MODELS: 
    if model not in ["qwen2_vl_qlora", "fatformer_official", "simple_cnn"]: 
        models_to_test.append(model)

artifact_test_metrics = evaluate_saved_models(models_to_test, X_test_artifact, y_test_artifact)
artifact_test_metrics = artifact_test_metrics.sort_values("accuracy", ascending=False)
print("\n🎯 ArtiFact Test Results:")
artifact_test_metrics

## 7. Test on CIFAKE Dataset  
Now test cross-dataset generalization on CIFAKE (32×32, Stable Diffusion only).


In [ ]:
# Download and prepare CIFAKE for testing
from utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_CIFAKE = "birdy654/cifake-real-and-ai-generated-synthetic-images"
CIFAKE_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_CIFAKE,
    local_dir=Path("data/cifake"),
    description="CIFAKE dataset",
    kaggle_subdir="cifake-real-and-ai-generated-synthetic-images",
)

if (CIFAKE_DIR / "test").exists():
    print(f"✅ CIFAKE data available at {CIFAKE_DIR}")
else:
    print(f"⚠️ CIFAKE dataset missing expected 'test' directory at {CIFAKE_DIR}")

In [ ]:
# CIFAKE uses 32x32 images, so we need different transform
CIFAKE_SIZE = 32
cifake_transform = transforms.Compose([
    transforms.Resize((CIFAKE_SIZE, CIFAKE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

cifake_test_dir = CIFAKE_DIR / "test"
if cifake_test_dir.exists():
    cifake_test_ds = datasets.ImageFolder(cifake_test_dir, transform=cifake_transform)
    X_test_cifake, y_test_cifake = dataset_to_numpy(cifake_test_ds)
    print(f"CIFAKE test shape: {X_test_cifake.shape}")
    print(f"CIFAKE classes: {cifake_test_ds.classes}")
    print(f"CIFAKE test class distribution: {np.bincount(y_test_cifake)}")
else:
    print("❌ CIFAKE test directory not found")

In [ ]:
# Test cross-dataset generalization: ArtiFact → CIFAKE
if 'X_test_cifake' in locals():
    cifake_test_metrics = evaluate_saved_models(models_to_test, X_test_cifake, y_test_cifake)
    cifake_test_metrics = cifake_test_metrics.sort_values("accuracy", ascending=False)
    print("\n🔄 Cross-Dataset Results: ArtiFact → CIFAKE")
    cifake_test_metrics
else:
    print("❌ CIFAKE test data not available")

## 8. Test on Shoes Dataset
Finally, test on the Shoes dataset (Midjourney generated).


In [ ]:
# Download and prepare Shoes Dataset
from utils.kaggle_utils import ensure_kaggle_dataset

# Kaggle dataset: https://www.kaggle.com/datasets/sunnykakar/shoes-dataset-real-and-ai-generated-images
KAGGLE_SHOES = "sunnykakar/shoes-dataset-real-and-ai-generated-images"

# ✅ Use the actual extracted top-level folder name from the dataset zip:
EXPECTED_TOPDIR = "shoes-dataset-real-and-ai-generated-images"

SHOES_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_SHOES,
    local_dir=Path("data/shoes-ai-vs-real"),
    description="SunnyKakar Shoes (AI vs Real)",
    kaggle_subdir=EXPECTED_TOPDIR,
)

if SHOES_DIR.exists():
    print(f"✅ Shoes dataset available at {SHOES_DIR}")
else:
    print(f"⚠️ Shoes dataset missing at {SHOES_DIR}")

In [ ]:
def find_imagefolder_split(base_dir: Path) -> Path:
    base_dir = Path(base_dir)

    # If the provided base is already an ImageFolder root, use it.
    def is_imagefolder_dir(path: Path) -> bool:
        path = Path(path)
        if not path.is_dir():
            return False
        subdirs = [p for p in path.iterdir() if p.is_dir()]
        if len(subdirs) < 2:
            return False
        # At least one class subdir must contain files
        return any(any(child.is_file() for child in d.iterdir()) for d in subdirs)
    
    if is_imagefolder_dir(base_dir):
        return base_dir

    # Common split folder names
    preferred = ("train", "test", "validation", "val", "eval", "holdout")
    for name in preferred:
        for variant in {name, name.upper(), name.capitalize()}:
            candidate = base_dir / variant
            if is_imagefolder_dir(candidate):
                return candidate

    # Extra: If dataset returned local_dir but the data is nested one level deeper,
    # look for a folder that contains both 'ai-midjourney' and 'real' with images inside.
    for candidate in base_dir.glob("*"):
        if candidate.is_dir():
            ai_dir = candidate / "ai-midjourney"
            real_dir = candidate / "real"
            if ai_dir.exists() and real_dir.exists() and is_imagefolder_dir(candidate):
                return candidate

    # General recursive search as a final fallback.
    for candidate in sorted(base_dir.rglob("*")):
        if is_imagefolder_dir(candidate):
            return candidate

    raise ValueError(f"Could not locate an ImageFolder split inside {base_dir}")

# Shoes dataset likely uses higher resolution, resize to match our training
shoes_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),  # Resize to 200x200 to match ArtiFact
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

try:
    shoes_split = find_imagefolder_split(SHOES_DIR)
    print(f"Using {shoes_split} for evaluation")
    
    shoes_ds = datasets.ImageFolder(shoes_split, transform=shoes_transform)
    if len(shoes_ds) == 0:
        raise ValueError(f"No samples detected under {shoes_split}")
    
    X_test_shoes, y_test_shoes = dataset_to_numpy(shoes_ds)
    print(f"Shoes test shape: {X_test_shoes.shape}")
    print(f"Shoes classes: {shoes_ds.classes}")
    print(f"Shoes test class distribution: {np.bincount(y_test_shoes)}")
    
except Exception as e:
    print(f"❌ Error loading shoes dataset: {e}")
    X_test_shoes = None

In [ ]:
# Test cross-dataset generalization: ArtiFact → Shoes
if X_test_shoes is not None:
    shoes_test_metrics = evaluate_saved_models(models_to_test, X_test_shoes, y_test_shoes)
    shoes_test_metrics = shoes_test_metrics.sort_values("accuracy", ascending=False)
    print("\n👠 Cross-Dataset Results: ArtiFact → Shoes")
    shoes_test_metrics
else:
    print("❌ Shoes test data not available")

## 9. Summary and Comparison
Compare performance across all three test scenarios.


In [ ]:
# Create comprehensive comparison table
summary_results = []

# Collect results from all three test scenarios
if 'artifact_test_metrics' in locals():
    for _, row in artifact_test_metrics.iterrows():
        summary_results.append({
            'dataset': 'ArtiFact Test',
            'model': row['model'],
            'accuracy': row['accuracy'],
            'f1_macro': row['f1_macro'],
            'roc_auc': row['roc_auc'],
            'pr_auc': row['pr_auc']
        })

if 'cifake_test_metrics' in locals():
    for _, row in cifake_test_metrics.iterrows():
        summary_results.append({
            'dataset': 'CIFAKE Test',
            'model': row['model'], 
            'accuracy': row['accuracy'],
            'f1_macro': row['f1_macro'],
            'roc_auc': row['roc_auc'],
            'pr_auc': row['pr_auc']
        })

if 'shoes_test_metrics' in locals():
    for _, row in shoes_test_metrics.iterrows():
        summary_results.append({
            'dataset': 'Shoes Test',
            'model': row['model'],
            'accuracy': row['accuracy'], 
            'f1_macro': row['f1_macro'],
            'roc_auc': row['roc_auc'],
            'pr_auc': row['pr_auc']
        })

summary_df = pd.DataFrame(summary_results)

if not summary_df.empty:
    # Create pivot table for easier comparison
    pivot_accuracy = summary_df.pivot(index='model', columns='dataset', values='accuracy')
    print("\n📊 ACCURACY COMPARISON ACROSS DATASETS:")
    print(pivot_accuracy.round(4))
    
    pivot_auc = summary_df.pivot(index='model', columns='dataset', values='roc_auc') 
    print("\n📈 ROC-AUC COMPARISON ACROSS DATASETS:")
    print(pivot_auc.round(4))
    
    # Calculate generalization gaps (if multiple datasets available)
    if len(summary_df['dataset'].unique()) > 1:
        print("\n🔍 GENERALIZATION ANALYSIS:")
        if 'ArtiFact Test' in pivot_accuracy.columns:
            for col in pivot_accuracy.columns:
                if col != 'ArtiFact Test':
                    gap = pivot_accuracy['ArtiFact Test'] - pivot_accuracy[col]
                    print(f"\nAccuracy Drop: ArtiFact → {col}")
                    print(gap.sort_values(ascending=False).round(4))
else:
    print("❌ No test results available for comparison")

In [ ]:
from pathlib import Path

if not summary_df.empty:
    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)
    output_path = results_dir / "artifact_cross_dataset_results.csv"
    summary_df.to_csv(output_path, index=False)
    print(f"✅ Results saved to: {output_path.resolve()}")

    # Display final summary
    print("\n🏆 FINAL SUMMARY:")
    print(f"Models trained: {len(models_to_test)}")
    print(f"Training dataset: ArtiFact (200×200, 25 generators)")
    print(f"Test datasets: {summary_df['dataset'].unique()}")

    best_model_per_dataset = summary_df.loc[summary_df.groupby('dataset')['accuracy'].idxmax()]
    print("\n🥇 Best model per dataset:")
    for _, row in best_model_per_dataset.iterrows():
        print(f"  {row['dataset']}: {row['model']} ({row['accuracy']:.4f} accuracy)")
    
    print(f"\n📥 Download results with:")
    print(f"  scp your-vm:{output_path.resolve()} .")
else:
    print("❌ No results to save")

## 10. Key Findings
Based on the results, we can analyze:

1. **In-Domain Performance**: How well models perform on ArtiFact test data
2. **Resolution Generalization**: Performance drop when testing on CIFAKE (200→32 pixels)
3. **Generator Generalization**: Performance drop when testing on Shoes (25 generators → Midjourney)
4. **Model Architecture Impact**: Which architectures generalize best across domains

The ArtiFact dataset's diversity (25 generators vs CIFAKE's 1 generator) should provide better 
cross-dataset generalization compared to the original CIFAKE-trained models.
